In [ ]:
import polars as pl
import geopandas as gpd
import numpy as np
from scipy.spatial.distance import cdist
import glob
import os
import shutil
import urllib.request
import zipfile
import gc
import holidays
from datetime import datetime
from shapely.geometry import Point

DATA_FOLDER = "nyc_data"
WEATHER_FILE = "nyc_weather.csv" 
TEMP_FOLDER = "temp_chunks"
OUTPUT_DIR = "stgcn_dataset"

FREQ = "1h" 
NUM_NODES = 263
ROLLING_WINDOW_DAYS = 28

VALID_START_DATE = datetime(2016, 1, 1, 0, 0, 0)
VALID_END_DATE = datetime(2025, 8, 26, 23, 59, 59)

os.makedirs(TEMP_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# 1. NY Holidays Calendar
def get_ny_holidays_df(start_year, end_year):
    ny_holidays = holidays.US(subdiv='NY', years=range(start_year, end_year + 1))
    dates = list(ny_holidays.keys())
    return pl.DataFrame({"date": dates, "is_holiday": 1}).with_columns(pl.col("date").cast(pl.Date))

# 2. Spatial Graph & Static Geographic Features
def build_spatial_and_geo_features(out_dir):
    zone_dir = "taxi_zones"
    
    shp_path = os.path.join(zone_dir, "taxi_zones.shp")
    gdf = gpd.read_file(shp_path)
    gdf = gdf[gdf['LocationID'].astype(int).isin(range(1, NUM_NODES + 1))]
    gdf = gdf.sort_values('LocationID')
    gdf = gdf.to_crs("EPSG:2263")
    
    # 1. Adjacency Matrix
    centroids = np.array(list(zip(gdf.geometry.centroid.x, gdf.geometry.centroid.y)))
    dist_matrix = cdist(centroids, centroids, metric='euclidean')
    std_dist = dist_matrix.std()
    A_spatial = np.exp(-(dist_matrix ** 2) / (std_dist ** 2))
    A_spatial[A_spatial < 0.1] = 0.0 
    np.fill_diagonal(A_spatial, 1.0)
    np.save(os.path.join(out_dir, "adj_spatial.npy"), A_spatial)
    
    # 2. Geography
    gdf['zone_area_sqkm'] = gdf.geometry.area * 9.2903e-8
    ts_point = gpd.GeoSeries([Point(-73.9851, 40.7589)], crs="EPSG:4326").to_crs("EPSG:2263").iloc[0]
    gdf['dist_to_center_km'] = gdf.geometry.centroid.distance(ts_point) * 0.0003048
    
    df_static_geo = pl.DataFrame({
        "LocationID": gdf['LocationID'].astype(int).tolist(),
        "zone_area_sqkm": gdf['zone_area_sqkm'].tolist(),
        "dist_to_center_km": gdf['dist_to_center_km'].tolist()
    })
    return df_static_geo

# 3. Flow Graph
def build_flow_adjacency_matrix(data_folder, out_dir):
    print("--- [2/6] Building Flow Adjacency Matrix ---")
    all_files = sorted(glob.glob(os.path.join(data_folder, "**/*.parquet")))
    all_chunks = []
    
    for file_path in all_files:
        try:
            q = pl.scan_parquet(file_path)
            cols = {c.lower(): c for c in q.columns}
            pu_col, do_col = cols.get("pulocationid"), cols.get("dolocationid")
            time_col = cols.get("tpep_pickup_datetime")
            if not pu_col or not do_col or not time_col: continue
            
            chunk = (
                q.select([
                    pl.col(time_col).cast(pl.Datetime).alias("time_bin"),
                    pl.col(pu_col).cast(pl.Int64).alias("PU"),
                    pl.col(do_col).cast(pl.Int64).alias("DO")
                ])
                .filter(
                    (pl.col("time_bin") >= VALID_START_DATE) & (pl.col("time_bin") <= VALID_END_DATE) &
                    (pl.col("PU") >= 1) & (pl.col("PU") <= NUM_NODES) &
                    (pl.col("DO") >= 1) & (pl.col("DO") <= NUM_NODES)
                )
                .group_by(["PU", "DO"]).agg(pl.len().alias("trips")).collect()
            )
            if len(chunk) > 0: all_chunks.append(chunk)
        except Exception as e: pass
            
    od_counts = pl.concat(all_chunks).group_by(["PU", "DO"]).agg(pl.col("trips").sum())
    A_flow = np.zeros((NUM_NODES, NUM_NODES), dtype=np.float32)
    
    pu_arr, do_arr, trips_arr = od_counts["PU"].to_numpy() - 1, od_counts["DO"].to_numpy() - 1, od_counts["trips"].to_numpy()
    A_flow[pu_arr, do_arr] = trips_arr
    row_sums = A_flow.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1 
    A_flow = A_flow / row_sums
    np.save(os.path.join(out_dir, "adj_flow.npy"), A_flow)

# 4. weather features
def process_weather(weather_csv, freq="1h"):
    q_weather = pl.scan_csv(weather_csv, ignore_errors=True)
    return (
        q_weather.with_columns([
            pl.col("DATE").cast(pl.Datetime).dt.truncate(freq).alias("time_bin"),
            pl.col("TMP").str.split(",").list.get(0).cast(pl.Float32, strict=False).alias("tmp"),
            pl.col("WND").str.split(",").list.get(3).cast(pl.Float32, strict=False).alias("wnd"),
            pl.col("AA1").fill_null("01,0000,9,5").str.split(",").list.get(1).cast(pl.Float32, strict=False).alias("precip")
        ])
        .filter((pl.col("time_bin") >= VALID_START_DATE) & (pl.col("time_bin") <= VALID_END_DATE))
        .with_columns([
            pl.when(pl.col("tmp") == 9999).then(None).otherwise(pl.col("tmp") / 10.0).alias("temperature"),
            pl.when(pl.col("wnd") == 9999).then(None).otherwise(pl.col("wnd") / 10.0).alias("wind_speed"),
            pl.when(pl.col("precip") == 9999).then(None).otherwise(pl.col("precip") / 10.0).alias("precipitation")
        ])
        .group_by("time_bin").agg([pl.col("temperature").mean(), pl.col("wind_speed").mean(), pl.col("precipitation").sum()])
        .sort("time_bin")
        .with_columns([
            pl.col("temperature").forward_fill().backward_fill(),
            pl.col("wind_speed").forward_fill().backward_fill(),
            pl.col("precipitation").fill_null(0.0)
        ])
    ).collect()

# 5. Hourly Taxi node feature
def map_taxi_features(data_folder, temp_folder, freq="1h"):
    all_files = sorted(glob.glob(os.path.join(data_folder, "**/*.parquet")))
    
    for i, file_path in enumerate(all_files):
        try:
            q = pl.scan_parquet(file_path)
            cols = [c.lower() for c in q.columns]
            time_col = next((c for c in q.columns if c.lower() == "tpep_pickup_datetime"), None)
            loc_col = next((c for c in q.columns if c.lower() == "pulocationid"), None)
            fare_col = next((c for c in q.columns if c.lower() == "fare_amount"), None)
            tip_col = next((c for c in q.columns if c.lower() == "tip_amount"), None)
            if not (time_col and loc_col): continue

            time_expr = pl.col(time_col)
            if q.schema[time_col] == pl.Int64: time_expr = pl.from_epoch(time_expr, time_unit="ms")
            else: time_expr = time_expr.cast(pl.Datetime)

            chunk = (
                q.select([
                    time_expr.dt.truncate(freq).alias("time_bin"),
                    pl.col(loc_col).cast(pl.Int64).alias("LocationID"),
                    pl.col(fare_col).cast(pl.Float32).alias("fare"),
                    pl.col(tip_col).cast(pl.Float32).alias("tip"),
                ])
                .filter(
                    (pl.col("time_bin") >= VALID_START_DATE) & (pl.col("time_bin") <= VALID_END_DATE) &
                    (pl.col("LocationID") >= 1) & (pl.col("LocationID") <= NUM_NODES) & (pl.col("fare") >= 0)
                )
                .group_by(["time_bin", "LocationID"])
                .agg([pl.len().alias("demand"), pl.col("fare").sum().alias("revenue_fare"), pl.col("tip").sum().alias("revenue_tip")])
            ).collect()
            
            if len(chunk) > 0: chunk.write_parquet(os.path.join(temp_folder, f"chunk_{i}.parquet"))
            del q, chunk
            gc.collect()
        except Exception as e: pass

# 6. Rolling Profiling
def build_rolling_proxies(temp_folder):    
    # 1. daily feature extraction
    q_taxi = pl.scan_parquet(os.path.join(temp_folder, "*.parquet"))
    
    daily_aggs = (
        q_taxi
        .with_columns([
            pl.col("time_bin").dt.date().alias("date"),
            pl.col("time_bin").dt.hour().alias("hour")
        ])
        .group_by(["date", "LocationID"])
        .agg([
            pl.col("demand").sum().alias("daily_trips"),
            pl.col("revenue_fare").sum().alias("daily_fare"),
            pl.col("revenue_tip").sum().alias("daily_tip"),
            # morning peak vs evening peak
            pl.col("demand").filter((pl.col("hour") >= 7) & (pl.col("hour") <= 10)).sum().alias("morning_trips"),
            pl.col("demand").filter((pl.col("hour") >= 16) & (pl.col("hour") <= 19)).sum().alias("evening_trips")
        ])
    ).collect()

    # 2. [date x node]
    min_date, max_date = daily_aggs["date"].min(), daily_aggs["date"].max()
    full_days = pl.datetime_range(min_date, max_date, interval="1d", eager=True).cast(pl.Date).alias("date").to_frame()
    all_locs = pl.int_range(1, NUM_NODES + 1, eager=True).alias("LocationID").to_frame()
    daily_grid = full_days.join(all_locs, how="cross")

    # 3. 28 days Rolling MAs
    rolling_proxies = (
        daily_grid.join(daily_aggs, on=["date", "LocationID"], how="left").fill_null(0)
        .sort(["LocationID", "date"])
        .with_columns([
            pl.col("daily_trips").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_trips"),
            pl.col("daily_fare").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_fare"),
            pl.col("daily_tip").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_tip"),
            pl.col("morning_trips").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_morning"),
            pl.col("evening_trips").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_evening"),
        ])
        .with_columns([
            (pl.col("roll_tip") / pl.col("roll_fare")).fill_nan(0.0).alias("rolling_tip_pct"),
            (pl.col("roll_fare") / pl.col("roll_trips")).fill_nan(0.0).alias("rolling_avg_fare"),
            (pl.col("roll_morning") / (pl.col("roll_evening") + 1)).fill_nan(0.0).alias("rolling_peak_ratio")
        ])
        # shift rolling feature to prevent data leakage
        .with_columns([
            pl.col("rolling_tip_pct").shift(1).over("LocationID"),
            pl.col("rolling_avg_fare").shift(1).over("LocationID"),
            pl.col("rolling_peak_ratio").shift(1).over("LocationID")
        ])
        .fill_null(strategy="backward")
        .select(["date", "LocationID", "rolling_tip_pct", "rolling_avg_fare", "rolling_peak_ratio"])
    )
    
    return rolling_proxies

# 7. Reduce & STGCN Matrix Alignment
def build_stgcn_dataset(temp_folder, df_weather, df_static_geo, df_rolling_proxies, df_holidays, out_dir, freq="1h"):    
    q_nodes = (
        pl.scan_parquet(os.path.join(temp_folder, "*.parquet"))
        .group_by(["time_bin", "LocationID"])
        .agg([pl.col("demand").sum(), pl.col("revenue_fare").sum(), pl.col("revenue_tip").sum()])
    ).collect()

    min_date, max_date = max(q_nodes["time_bin"].min(), VALID_START_DATE), min(q_nodes["time_bin"].max(), VALID_END_DATE)
    full_timeline = pl.datetime_range(min_date, max_date, interval=freq, eager=True).alias("time_bin").to_frame()
    all_locations = pl.int_range(1, NUM_NODES + 1, eager=True).alias("LocationID").to_frame()
    
    lz_ml = (
        full_timeline.lazy().join(all_locations.lazy(), how="cross")
        .join(q_nodes.lazy(), on=["time_bin", "LocationID"], how="left")
        .with_columns([
            pl.col("demand").fill_null(0),
            pl.col("revenue_fare").fill_null(0.0),
            pl.col("revenue_tip").fill_null(0.0),
        ])
        .with_columns((pl.col("revenue_fare") + pl.col("revenue_tip")).alias("revenue_total"))
        
        # 1. weather
        .join(df_weather.lazy(), on="time_bin", how="left")
        # 2. geography (area, center distance)
        .join(df_static_geo.lazy(), on="LocationID", how="left")
        
        # 3. rolling daily econ feature
        .with_columns(pl.col("time_bin").dt.date().alias("date"))
        .join(df_rolling_proxies.lazy(), on=["date", "LocationID"], how="left")
        
        # 4. holiday
        .join(df_holidays.lazy(), on="date", how="left")
        .with_columns(pl.col("is_holiday").fill_null(0).cast(pl.Int32))
        .drop("date")
        
        # node_index (LocationID - 1)
        .with_columns((pl.col("LocationID") - 1).alias("node_index"))
        .sort(["time_bin", "node_index"])
        
        # 5. time-wise data
        .with_columns([
            pl.col("time_bin").dt.hour().alias("hour"),
            pl.col("time_bin").dt.weekday().alias("weekday"),
            
            np.sin(2 * np.pi * pl.col("time_bin").dt.hour() / 24).alias("hour_sin"),
            np.cos(2 * np.pi * pl.col("time_bin").dt.hour() / 24).alias("hour_cos"),
            np.sin(2 * np.pi * pl.col("time_bin").dt.weekday() / 7).alias("weekday_sin"),
            np.cos(2 * np.pi * pl.col("time_bin").dt.weekday() / 7).alias("weekday_cos"),

            pl.col("demand").shift(1).over("node_index").alias("demand_lag_1"),
            pl.col("demand").shift(24).over("node_index").alias("demand_lag_24"),
            pl.col("demand").shift(168).over("node_index").alias("demand_lag_168"),
            
            pl.col("revenue_total").shift(1).over("node_index").alias("revenue_lag_1"),
            pl.col("revenue_total").shift(24).over("node_index").alias("revenue_lag_24"),
            pl.col("revenue_total").shift(168).over("node_index").alias("revenue_lag_168"),
        ])
        .filter(pl.col("demand_lag_168").is_not_null())
    )
    
    out_file = os.path.join(out_dir, "node_features_X.parquet")
    lz_ml.sink_parquet(out_file)
    print(f"✅ Final Node Feature Tensor Grid saved to: {out_file}")

In [ ]:
df_holidays = get_ny_holidays_df(2016, 2025)
df_static_geo = build_spatial_and_geo_features(OUTPUT_DIR)

build_flow_adjacency_matrix(DATA_FOLDER, OUTPUT_DIR)
df_weather_agg = process_weather(WEATHER_FILE, freq=FREQ)

map_taxi_features(DATA_FOLDER, TEMP_FOLDER, freq=FREQ)
df_rolling_proxies = build_rolling_proxies(TEMP_FOLDER)
build_stgcn_dataset(TEMP_FOLDER, df_weather_agg, df_static_geo, df_rolling_proxies, df_holidays, OUTPUT_DIR, freq=FREQ)

shutil.rmtree(TEMP_FOLDER, ignore_errors=True)

--- Generating NY Holidays Calendar ---
--- [1/6] Building Spatial Graph & Static Geographic Features ---
--- [2/6] Building Flow Adjacency Matrix ---


C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:90: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = {c.lower(): c for c in q.columns}


--- [3/6] Processing Weather Features ---
--- [4/6] Mapping Taxi Node Features into Chunks ---


C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:159: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  cols = [c.lower() for c in q.columns]
C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:160: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  time_col = next((c for c in q.columns if c.lower() == "tpep_pickup_datetime"), None)
C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:161: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without th

--- [5/6] Calculating Dynamic Socio-Economic Profiles (28-Day Rolling) ---


C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:227: DeprecationWarning: the argument `min_periods` for `Expr.rolling_sum` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col("daily_trips").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_trips"),
C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:228: DeprecationWarning: the argument `min_periods` for `Expr.rolling_sum` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col("daily_fare").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_fare"),
C:\Users\zhanq\AppData\Local\Temp\ipykernel_20304\3246022750.py:229: DeprecationWarning: the argument `min_periods` for `Expr.rolling_sum` is deprecated. It was renamed to `min_samples` in version 1.21.0.
  pl.col("daily_tip").rolling_sum(window_size=ROLLING_WINDOW_DAYS, min_periods=1).over("LocationID").alias("roll_tip"),
C:\Users\zhanq\AppD

--- [6/6] Assembling the Ultimate Spatio-Temporal Tensor Grid ---
✅ Final Node Feature Tensor Grid saved to: stgcn_dataset\node_features_X.parquet

🎉 ALL DONE! State-of-the-Art STGCN Dataset is ready!


In [16]:
df = pl.read_parquet("stgcn_dataset/node_features_X.parquet", n_rows=10)

In [17]:
df

time_bin,LocationID,demand,revenue_fare,revenue_tip,revenue_total,temperature,wind_speed,precipitation,zone_area_sqkm,dist_to_center_km,rolling_tip_pct,rolling_avg_fare,rolling_peak_ratio,is_holiday,node_index,hour,weekday,hour_sin,hour_cos,weekday_sin,weekday_cos,demand_lag_1,demand_lag_24,demand_lag_168,revenue_lag_1,revenue_lag_24,revenue_lag_168
datetime[μs],i64,u32,f32,f32,f32,f32,f32,f32,f64,f64,f32,f64,f64,i32,i64,i8,i8,f64,f64,f64,f64,u32,u32,u32,f32,f32,f32
2016-01-08 00:00:00,1,0,0.0,0.0,0.0,5.6,0.0,0.0,7.343009,17.611394,0.125626,71.43822,0.411765,0,0,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
2016-01-08 00:00:00,2,0,0.0,0.0,0.0,5.6,0.0,0.0,13.369627,20.450277,0.192119,29.5,0.0,0,1,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
2016-01-08 00:00:00,3,0,0.0,0.0,0.0,5.6,0.0,0.0,2.943639,16.50427,0.0,19.142857,1.0,0,2,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
2016-01-08 00:00:00,4,50,599.0,77.610001,676.609985,5.6,0.0,0.0,0.745429,3.963087,0.121164,11.541365,1.383575,0,3,0,5,0.0,1.0,-0.974928,-0.222521,63,38,153,913.75,551.210022,2041.109985
2016-01-08 00:00:00,5,0,0.0,0.0,0.0,5.6,0.0,0.0,4.683694,28.642288,0.0,90.0,0.0,0,4,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
2016-01-08 00:00:00,6,0,0.0,0.0,0.0,5.6,0.0,0.0,3.802965,19.072997,0.042879,21.416667,2.0,0,5,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
2016-01-08 00:00:00,7,42,509.299988,22.900002,532.200012,5.6,0.0,0.0,3.654901,5.530281,0.097591,11.930353,1.714801,0,6,0,5,0.0,1.0,-0.974928,-0.222521,25,37,115,305.279999,347.809998,1324.220093
2016-01-08 00:00:00,8,0,0.0,0.0,0.0,5.6,0.0,0.0,0.24924,5.672595,0.142512,27.861111,1.8,0,7,0,5,0.0,1.0,-0.974928,-0.222521,0,0,1,0.0,0.0,9.86
2016-01-08 00:00:00,9,0,0.0,0.0,0.0,5.6,0.0,0.0,3.173958,16.671432,0.200308,10.833333,0.0,0,8,0,5,0.0,1.0,-0.974928,-0.222521,0,0,0,0.0,0.0,0.0
